# MindText - text pipeline walkthrough (SYNTHETIC data)

This notebook calls the packaged code on the synthetic sample. **Metrics here are meaningless**; they only show the workflow. It replaces the Reddit notebooks' logic (`notebooks/original/`) with the leakage-safe pipeline. Outputs are not stored (cells were executed as a script during repository assembly; the notebook itself was not run in Jupyter).

> Experimental research output - not a diagnosis.

In [ ]:
import tempfile, pathlib
from mindtext.config import Config
from mindtext.synthetic import make_text_dataset
from mindtext.training import run_experiment
from mindtext.reporting import format_report_text

work = pathlib.Path(tempfile.mkdtemp())
make_text_dataset(40, seed=0)[['id','text','label']].to_csv(work/'sample.csv', index=False)

In [ ]:
cfg = Config.from_yaml('../../configs/default.yaml') if pathlib.Path('../../configs/default.yaml').exists() else Config.from_yaml('configs/default.yaml')
cfg.output.reports_dir = str(work/'reports'); cfg.output.artifacts_dir = str(work/'artifacts')
res = run_experiment(cfg, data_path=work/'sample.csv', log=lambda m: None)
print(format_report_text(res.report))

In [ ]:
from mindtext.persistence import load_bundle
bundle = load_bundle(res.model_path)
bundle.predict_frame(['worry about the exam deadline', 'so tired and empty lately'])

## Next steps with the real data
1. `python -m mindtext prepare-reddit --raw-dir <raw> --out data/reddit_mental_health.csv`
2. `python -m mindtext data-check --data data/reddit_mental_health.csv --config configs/default.yaml`
3. Add `group_column`/`time_column` to the config if the source provides them and switch `split.strategy` to `grouped`/`chronological`.
4. `python -m mindtext train --config configs/default.yaml --data data/reddit_mental_health.csv`